# Silver Layer: Route Table Cleaning & Validation

Cleanses `skyledger_airlines.bronze.route` into `skyledger_airlines.silver.route`.
Applies primary key validation (§3.1) and numeric range validation (§3.2 extension).
Invalid records are quarantined in `silver.route_quarantine` with comma-separated rejection reasons.
Includes Bronze-Silver-Quarantine reconciliation and a data-quality report.

**Relationships:** `route` → `flight_booking` (product sold on each segment).
`airline_route` joins `airline` and `route` (which carriers operate which routes).

**FK note (§3.4):** The business spec references `rt_originnationkey` and `rt_destnationkey`,
but this dataset's route table does not contain origin/destination nation key columns.
The route name encodes airport pairs (e.g., `PEK-ADD`). The FK-to-nation check is therefore
not applicable to this table. **Numeric rules:** `ro_baseprice` must be strictly greater than zero; `ro_capacity` must be a positive integer (1 or more).

In [0]:
%sql
select * from skyledger_airlines.bronze.route

In [0]:
%sql
SELECT *
FROM skyledger_airlines.silver.route
WHERE ro_baseprice <= 0
   OR ro_capacity <= 0;

In [0]:
# ── Step 1: Read Bronze, Cast Types, Drop Metadata Columns ──
from pyspark.sql.functions import col, when, lit, count as _count, concat_ws
from pyspark.sql.window import Window

bronze_route = spark.table("skyledger_airlines.bronze.route")

# Drop metadata columns: _ingestion_timestamp, _source_file, _pipeline_run_id
# Cast key/numeric columns; keep text columns as STRING
route_typed = bronze_route.select(
    col("ro_routeid").cast("int").alias("ro_routeid"),
    col("ro_name").cast("string").alias("ro_name"),
    col("ro_aircraft").cast("string").alias("ro_aircraft"),
    col("ro_airline_brand").cast("string").alias("ro_airline_brand"),
    col("ro_routetype").cast("string").alias("ro_routetype"),
    col("ro_capacity").cast("int").alias("ro_capacity"),
    col("ro_cabinconfig").cast("string").alias("ro_cabinconfig"),
    col("ro_baseprice").cast("double").alias("ro_baseprice"),
    col("ro_comment").cast("string").alias("ro_comment"),
)

bronze_count = route_typed.count()
print(f"Bronze route rows read: {bronze_count}")
display(route_typed.limit(10))

In [0]:
# ── Step 2: Primary Key Validation (§3.1) ──
# Rule: ro_routeid must be NOT NULL and UNIQUE.
# Failure codes: NULL_PK, DUPLICATE_PK

route_flagged = route_typed.withColumn(
    "is_null_pk",
    when(col("ro_routeid").isNull(), lit(True)).otherwise(lit(False)),
)

# Window-based duplicate detection: keep only the first occurrence in Silver, rest go to quarantine
pk_window = Window.partitionBy("ro_routeid").orderBy("ro_routeid")
route_flagged = route_flagged.withColumn(
    "_pk_count", _count("*").over(Window.partitionBy("ro_routeid"))
).withColumn(
    "pk_rank", _count("*").over(pk_window).cast("int")
)
route_flagged = route_flagged.withColumn(
    "is_duplicate_pk",
    when((col("ro_routeid").isNotNull()) & (col("_pk_count") > 1) & (col("pk_rank") > 1), lit(True)).otherwise(lit(False)),
)

null_pk_count = route_flagged.filter(col("is_null_pk")).count()
duplicate_pk_count = route_flagged.filter(col("is_duplicate_pk")).count()

print(f"NULL_PK failures:      {null_pk_count}")
print(f"DUPLICATE_PK failures: {duplicate_pk_count}")

In [0]:
# ── Step 2b: Numeric Range Validation (§3.2 extension) ──
# Rule: ro_baseprice must be strictly greater than zero.
# Rule: ro_capacity must be a positive integer (1 or more).
# Failure codes: INVALID_BASEPRICE, INVALID_CAPACITY

route_flagged = route_flagged.withColumn(
    "is_invalid_baseprice",
    when(col("ro_baseprice").isNull() | (col("ro_baseprice") <= 0), lit(True)).otherwise(lit(False)),
).withColumn(
    "is_invalid_capacity",
    when(col("ro_capacity").isNull() | (col("ro_capacity") <= 0), lit(True)).otherwise(lit(False)),
)

invalid_baseprice_count = route_flagged.filter(col("is_invalid_baseprice")).count()
invalid_capacity_count = route_flagged.filter(col("is_invalid_capacity")).count()

print(f"INVALID_BASEPRICE failures: {invalid_baseprice_count}")
print(f"INVALID_CAPACITY failures:  {invalid_capacity_count}")

In [0]:
# ── Step 3: Build Rejection Reasons, Split Clean vs Quarantine, Write Tables ──

# Build comma-separated rejection_reason from all flag columns
route_flagged = route_flagged.withColumn(
    "rejection_reason",
    concat_ws(
        ", ",
        when(col("is_null_pk"), lit("NULL_PK")),
        when(col("is_duplicate_pk"), lit("DUPLICATE_PK")),
        when(col("is_invalid_baseprice"), lit("INVALID_BASEPRICE")),
        when(col("is_invalid_capacity"), lit("INVALID_CAPACITY")),
    ),
)

# Clean records: passed every check (rejection_reason is empty string)
silver_route = route_flagged.filter(col("rejection_reason") == "").select(
    "ro_routeid", "ro_name", "ro_aircraft", "ro_airline_brand",
    "ro_routetype", "ro_capacity", "ro_cabinconfig", "ro_baseprice", "ro_comment",
)

# Quarantined records: failed one or more checks
quarantine_route = route_flagged.filter(col("rejection_reason") != "").select(
    "ro_routeid", "ro_name", "ro_aircraft", "ro_airline_brand",
    "ro_routetype", "ro_capacity", "ro_cabinconfig", "ro_baseprice", "ro_comment",
    "rejection_reason",
)

silver_count = silver_route.count()
quarantine_count = quarantine_route.count()

print(f"Silver rows:     {silver_count}")
print(f"Quarantine rows: {quarantine_count}")

# Write Silver table (overwrite for idempotency — same result on every run)
(silver_route.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("skyledger_airlines.silver.route"))

# Write Quarantine table (overwrite for idempotency)
(quarantine_route.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("skyledger_airlines.silver.route_quarantine"))

print("✅ Tables written: silver.route, silver.route_quarantine")

In [0]:
# ── Step 4: Bronze-Silver-Quarantine Reconciliation (§3.5) ──
# Bronze row count must equal Silver + Quarantine for the route table.

bronze_total = spark.table("skyledger_airlines.bronze.route").count()
silver_total = spark.table("skyledger_airlines.silver.route").count()
quarantine_total = spark.table("skyledger_airlines.silver.route_quarantine").count()

print(f"Bronze row count:      {bronze_total}")
print(f"Silver row count:      {silver_total}")
print(f"Quarantine row count:  {quarantine_total}")
print(f"Silver + Quarantine:   {silver_total + quarantine_total}")

assert bronze_total == silver_total + quarantine_total, (
    f"RECONCILIATION FAILED: Bronze ({bronze_total}) != "
    f"Silver ({silver_total}) + Quarantine ({quarantine_total})"
)

print("✅ Reconciliation PASSED: Bronze = Silver + Quarantine")

In [0]:
# ── Step 5: Data Quality Report ──
# Documents every check executed, even when zero rows fail.
# Note: §3.4 FK-to-nation check not applicable — route table has no
# origin/destination nation key columns in this dataset.

dq_report = spark.createDataFrame(
    [
        ("route", "NULL_PK", "ro_routeid must not be null", null_pk_count, "Quarantine"),
        ("route", "DUPLICATE_PK", "ro_routeid must be unique across all rows", duplicate_pk_count, "Quarantine"),
        ("route", "INVALID_BASEPRICE", "ro_baseprice must be strictly greater than zero", invalid_baseprice_count, "Quarantine"),
        ("route", "INVALID_CAPACITY", "ro_capacity must be a positive integer (1 or more)", invalid_capacity_count, "Quarantine"),
    ],
    ["table", "rule_code", "check_description", "failing_rows", "action_taken"],
)

display(dq_report)

# Show quarantined records if any exist
if quarantine_count > 0:
    print(f"\n── Quarantined Records ({quarantine_count} rows) ──")
    display(spark.table("skyledger_airlines.silver.route_quarantine"))
else:
    print("\nNo quarantined records — all route rows passed every check.")

# Show final Silver table
print(f"\n── Silver Route Table ({silver_count} rows) ──")
display(spark.table("skyledger_airlines.silver.route"))